# Somo 05 - RAG ya Wakala


## Usanidi

Daftari hili linaonyesha mfano wa Agentic RAG (Retrieval-Augmented Generation) kwa kutumia Microsoft Agent Framework.

**Mambo yanayohitajika:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — anwani ya huduma yako ya Azure AI Search
- `AZURE_SEARCH_API_KEY` — funguo yako ya API ya Azure AI Search
- Utekelezaji wa Azure OpenAI umewekwa kupitia mabadiliko ya mazingira
- Azure CLI imethibitishwa (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Nini Agentic RAG?

RAG ya kawaida inafuata mnyororo wa kazi uliowekwa: tafuta hati, kisha tengeneza jibu. **Agentic RAG** inaenda hatua zaidi kwa kumpa wakala uhuru wa kuamua **lini** na **jinsi** ya kupata taarifa.

Kwa Agentic RAG, wakala anaweza:
- **Kuamua** ikiwa upatikanaji unahitajika kabla ya kujibu swali
- **Kuchagua** chanzo cha data au zana ya kuulizia
- **Kutathmini** matokeo yaliyopatikana na kufanya upatikanaji wa ziada ikiwa jaribio la kwanza halitoshi
- **Kuchanganya** taarifa kutoka hatua mbalimbali za upatikanaji kuwa jibu linaloeleweka

Hii inafanya wakala kuwa na kubadilika zaidi na usahihi ikilinganishwa na mnyororo wa kazi wa kupata kisha kutengeneza majibu uliowekwa.


## Kuunda Zana ya Utafutaji

Katika Agentic RAG, vyanzo vya data vya nje vimefungwa kama **zana** ambazo wakala anaweza kuitumia kwa ombi. Hii inamwezesha wakala kutibu upokeaji kama kitendo kingine anachoweza kuchukua, badala ya hatua ya lazima.

Hapo chini tunaelezea hifadhidata ya maarifa ya safari na kuionyesha kama zana ambayo wakala anaweza kuitumia kutafuta taarifa za maeneo ya kusafari.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Kujenga Wakalishaji RAG

Sasa tunaunda wakala ambaye ameagizwa **daima kupata taarifa kabla ya kujibu**. Wakala anatumia zana ya `search_travel_knowledge` kupata majibu yake kwa msingi wa hifadhidata ya maarifa badala ya kutegemea data yake mwenyewe ya mafunzo.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Urejeshaji wa Kurudia — Mfumo wa Mtengenezaji-Mhukumu

Ulelemavu muhimu wa Agentic RAG ni **urejeshaji wa kurudia**. Wakala anaweza kufanya mizunguko kadhaa ya utafutaji kuthibitisha, kusafisha, au kupanua matokeo yake ya awali — sawa na mtiririko wa kazi wa "mtengenezaji-mhukumu":

1. **Hatua ya Mtengenezaji**: Wakala anarudisha taarifa za awali na kuandika jibu.
2. **Hatua ya Mhukumu**: Wakala hufanya urejeshaji za ziada kuthibitisha maelezo au kujaza mapengo.

Hapo chini, wakala anaulizwa swali linalohitaji kulinganisha maeneo kadhaa, jambo linalomtia moyo kutafuta mara kadhaa.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Muhtasari

Katika somo hili ulijifunza jinsi ya kujenga mfumo wa **Agentic RAG** kwa kutumia Microsoft Agent Framework:

- **Agentic RAG** huruhusu maagent kuvuna taarifa kwa uhuru wakati wowote, na kufanya uvunaji taarifa kuwa wa mabadiliko badala ya wa kudumu.
- **Zana kama vyanzo vya data**: Hifadhidata za maarifa za nje (kama Azure AI Search) zimefungwa kama zana ambazo maagent wanaweza kuitumia.
- **Uvunjaji wa mara kwa mara**: Mfano wa maker-checker huruhusu maagent kufanya mizunguko mingi ya uvunaji — kutafuta, kuthibitisha, na kuboresha — kabla ya kutoa jibu la mwisho.

Katika uzalishaji, ungebadilisha `TRAVEL_KNOWLEDGE_BASE` iliyo kwenye kumbukumbu ya ndani na faharasa halisi ya Azure AI Search kushughulikia uvunaji wa hati kubwa za usafiri.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Angalizo**:  
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kwa usahihi, tafadhali fahamu kuwa tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu. Hati ya awali katika lugha yake asilia inapaswa kuchukuliwa kama chanzo halali. Kwa taarifa muhimu, tafsiri ya kitaalamu inayotolewa na watu inashauriwa. Sisi hatuwezi kuwajibika kwa ukiukwaji wowote wa maana au tafsiri potovu zinazotokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
